# 04 — IDD XML-to-YOLO Conversion

Project: Privacy-Preserving Edge AI Framework for Intelligent Transportation

Total cells: 10 (1 Markdown + 9 Code)

Objectives:
1. Load saved dataset manifests and class metadata.
2. Establish a consistent class-to-ID mapping.
3. Convert Pascal VOC XML annotations to YOLO format.
4. Save converted labels for training and validation.
5. Generate a reusable dataset.yaml configuration.

Outputs:
- configs/class_mapping.json
- configs/dataset.yaml
- Prepared YOLO label files

Important:
- The test split has no annotations and will not receive fabricated labels.
- Original images and XML annotations will remain unchanged.
- Converted labels and configurations will be reused by the training notebooks.

In [1]:
# CELL 2: Imports and paths
from pathlib import Path
import json
import xml.etree.ElementTree as ET
from collections import Counter

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")
CONFIG = PROJECT / "configs"

metadata = json.loads((CONFIG / "idd_metadata.json").read_text())
manifest_path = CONFIG / "idd_preprocessing_manifest.csv"

print("Dataset:", ROOT)
print("Project:", PROJECT)
print("Manifest:", manifest_path)

Dataset: /workspace/IDD_Detection
Project: /workspace/ITS_Project
Manifest: /workspace/ITS_Project/configs/idd_preprocessing_manifest.csv


In [2]:
# CELL 3: Load reusable manifest
import pandas as pd

manifest = pd.read_csv(manifest_path)

train_manifest = manifest[manifest["split"] == "train"].copy()
val_manifest = manifest[manifest["split"] == "val"].copy()

print("Training images:", len(train_manifest))
print("Validation images:", len(val_manifest))
print("Test images:", len(manifest[manifest["split"] == "test"]))

Training images: 31569
Validation images: 10225
Test images: 4794


In [3]:
# CELL 4: Establish consistent class IDs
classes = sorted(metadata["classes"])
class_to_id = {name: idx for idx, name in enumerate(classes)}

mapping_path = CONFIG / "class_mapping.json"
mapping_path.write_text(json.dumps(class_to_id, indent=2))

print("Classes:", len(class_to_id))
print("Mapping saved:", mapping_path)
print(class_to_id)

Classes: 15
Mapping saved: /workspace/ITS_Project/configs/class_mapping.json
{'animal': 0, 'autorickshaw': 1, 'bicycle': 2, 'bus': 3, 'car': 4, 'caravan': 5, 'motorcycle': 6, 'person': 7, 'rider': 8, 'traffic light': 9, 'traffic sign': 10, 'trailer': 11, 'train': 12, 'truck': 13, 'vehicle fallback': 14}


In [4]:
# CELL 5: Prepare YOLO label directories
YOLO_DATA = PROJECT / "prepared_dataset"
LABELS = YOLO_DATA / "labels"

for split in ["train", "val"]:
    (LABELS / split).mkdir(parents=True, exist_ok=True)

print("Prepared dataset:", YOLO_DATA)
print("Training labels:", LABELS / "train")
print("Validation labels:", LABELS / "val")

Prepared dataset: /workspace/ITS_Project/prepared_dataset
Training labels: /workspace/ITS_Project/prepared_dataset/labels/train
Validation labels: /workspace/ITS_Project/prepared_dataset/labels/val


In [5]:
# CELL 6: Convert Pascal VOC bounding boxes to YOLO labels

def convert_xml_to_yolo(xml_path, output_path):
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    width = float(size.findtext("width"))
    height = float(size.findtext("height"))

    lines = []

    for obj in root.findall("object"):
        class_name = obj.findtext("name", default="unknown").strip()

        if class_name not in class_to_id:
            continue

        box = obj.find("bndbox")
        if box is None:
            continue

        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        # Clip coordinates to image boundaries
        xmin = max(0, min(xmin, width))
        xmax = max(0, min(xmax, width))
        ymin = max(0, min(ymin, height))
        ymax = max(0, min(ymax, height))

        if xmax <= xmin or ymax <= ymin:
            continue

        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        box_width = (xmax - xmin) / width
        box_height = (ymax - ymin) / height

        lines.append(
            f"{class_to_id[class_name]} "
            f"{x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + ("\n" if lines else ""))

    return len(lines)

In [6]:
# CELL 7: Generate YOLO labels for the training split

conversion_counts = {}

for _, row in train_manifest.iterrows():
    entry = row["entry"]
    xml_path = Path(row["annotation_path"])
    label_path = LABELS / "train" / f"{Path(entry).stem}.txt"

    if not xml_path.is_file():
        continue

    conversion_counts[str(label_path)] = convert_xml_to_yolo(
        xml_path, label_path
    )

print("Training label files created:", len(conversion_counts))
print("Training objects converted:", sum(conversion_counts.values()))

Training label files created: 4955
Training objects converted: 119003


In [7]:
# CELL 8: Generate YOLO labels for the validation split

val_conversion_counts = {}

for _, row in val_manifest.iterrows():
    entry = row["entry"]
    xml_path = Path(row["annotation_path"])
    label_path = LABELS / "val" / f"{Path(entry).stem}.txt"

    if not xml_path.is_file():
        continue

    val_conversion_counts[str(label_path)] = convert_xml_to_yolo(
        xml_path, label_path
    )

print("Validation label files created:", len(val_conversion_counts))
print("Validation objects converted:", sum(val_conversion_counts.values()))

Validation label files created: 3449
Validation objects converted: 73438


In [8]:
# CELL 9: Create a reusable Ultralytics dataset configuration

import yaml

dataset_config = {
    "path": str(YOLO_DATA),
    "train": "images/train",
    "val": "images/val",
    "names": {idx: name for name, idx in class_to_id.items()}
}

dataset_yaml = CONFIG / "dataset.yaml"
dataset_yaml.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True)
)

print("Dataset configuration saved:", dataset_yaml)
print(dataset_yaml.read_text())

Dataset configuration saved: /workspace/ITS_Project/configs/dataset.yaml
path: /workspace/ITS_Project/prepared_dataset
train: images/train
val: images/val
names:
  0: animal
  1: autorickshaw
  2: bicycle
  3: bus
  4: car
  5: caravan
  6: motorcycle
  7: person
  8: rider
  9: traffic light
  10: traffic sign
  11: trailer
  12: train
  13: truck
  14: vehicle fallback



In [9]:
# CELL 10: Save conversion summary for downstream notebooks

conversion_summary = {
    "dataset_root": str(ROOT),
    "prepared_dataset": str(YOLO_DATA),
    "class_mapping_path": str(mapping_path),
    "dataset_yaml": str(dataset_yaml),
    "classes": len(class_to_id),
    "train_label_files": len(conversion_counts),
    "train_objects_converted": sum(conversion_counts.values()),
    "val_label_files": len(val_conversion_counts),
    "val_objects_converted": sum(val_conversion_counts.values()),
    "test_annotations_available": False
}

summary_path = CONFIG / "yolo_conversion_summary.json"
summary_path.write_text(json.dumps(conversion_summary, indent=2))

print(json.dumps(conversion_summary, indent=2))

{
  "dataset_root": "/workspace/IDD_Detection",
  "prepared_dataset": "/workspace/ITS_Project/prepared_dataset",
  "class_mapping_path": "/workspace/ITS_Project/configs/class_mapping.json",
  "dataset_yaml": "/workspace/ITS_Project/configs/dataset.yaml",
  "classes": 15,
  "train_label_files": 4955,
  "train_objects_converted": 119003,
  "val_label_files": 3449,
  "val_objects_converted": 73438,
  "test_annotations_available": false
}


In [11]:
# CELL 11: Link original images into the prepared YOLO dataset
import os
from pathlib import Path

ROOT = Path("/workspace/IDD_Detection")
YOLO_DATA = Path("/workspace/ITS_Project/prepared_dataset")

for split in ["train", "val"]:
    image_dir = YOLO_DATA / "images" / split
    image_dir.mkdir(parents=True, exist_ok=True)

    entries = [
        line.strip()
        for line in (ROOT / f"{split}.txt").read_text().splitlines()
        if line.strip()
    ]

    linked = 0
    existing = 0
    missing = 0

    for entry in entries:
        source = (ROOT / "JPEGImages" / entry).with_suffix(".jpg")
        destination = image_dir / f"{Path(entry).stem}.jpg"

        if not source.is_file():
            missing += 1
            continue

        if destination.exists() or destination.is_symlink():
            existing += 1
            continue

        os.symlink(source, destination)
        linked += 1

    print(
        f"{split.upper()}: {linked} linked, "
        f"{existing} already present, {missing} missing"
    )

TRAIN: 4955 linked, 26614 already present, 0 missing
VAL: 3449 linked, 6776 already present, 0 missing
